# Trabalho Grau B — Classificação de Cenas Sonoras com CNNs

**Disciplina:** Inteligência Artificial e Aprendizado de Máquina (2026/1)  
**Professor:** Gabriel de Oliveira Ramos  
**Relatório Parcial — Entrega: 17/05/2025**

---

## 1. Introdução e Descrição do Problema

O problema abordado neste trabalho é a **classificação multi-label de sons ambientes** (*Sound Scene Classification*). Dado um arquivo de áudio, o sistema deve identificar quais classes de sons estão presentes simultaneamente naquela gravação — por exemplo, reconhecer que um áudio contém ao mesmo tempo o som de uma campainha e de um ventilador.

O dataset utilizado é proveniente do desafio **DCASE 2025** (*Detection and Classification of Acoustic Scenes and Events*), um benchmark internacional dedicado à análise e classificação de cenas acústicas.

### Por que CNNs para áudio?

Redes convolucionais (CNNs) foram originalmente desenvolvidas para processamento de imagens. A ponte entre o domínio de áudio e CNNs é feita por meio da conversão do sinal sonoro em uma representação visual 2D chamada **espectrograma de Mel** (frequência × tempo). Essa transformação transforma o problema de classificação de áudio em um problema de **classificação de imagens**, tornando direta a aplicação de CNNs.

Essa abordagem é consolidada na literatura e supera alternativas como alimentar áudio bruto à rede, pois o espectrograma já codifica as features acústicas mais relevantes — distribuição de energia por banda de frequência ao longo do tempo — economizando capacidade de representação da rede e acelerando o treinamento.

### Natureza multi-label

É importante destacar que este é um problema **multi-label** (e não multi-classe): em um mesmo áudio podem coexistir sons de múltiplas categorias simultaneamente. Isso impacta diretamente as escolhas de arquitetura e função de perda, conforme detalhado nas seções seguintes.

## 2. Dataset

### 2.1 Visão Geral

O dataset utilizado é o **DCASE 2025 — Task: Spatial Semantic Segmentation of Sound Scenes**, disponível em [https://dcase.community/challenge2025/](https://dcase.community/challenge2025/). O conjunto de dados contém **2.290 arquivos de áudio** distribuídos em **18 classes** de sons distintos. O objetivo é identificar, para cada arquivo, quais classes de sons estão presentes.

Como exemplos das categorias presentes no dataset, temos sons de **campainha**, **secador de cabelo** e **ventilador**, entre outros.

### 2.2 Classes Selecionadas

*(🔲 A ser preenchido pelo colega — justificativa da seleção das 5 classes: quais foram escolhidas, critério de seleção, distribuição de amostras por classe escolhida)*

Das 18 classes disponíveis, foram selecionadas **5 classes** para este trabalho. Os critérios de seleção levaram em conta: ...

| Classe | Quantidade de Amostras | % do Total Selecionado |
|--------|----------------------|------------------------|
| ...    | ...                  | ...                    |
| ...    | ...                  | ...                    |
| ...    | ...                  | ...                    |
| ...    | ...                  | ...                    |
| ...    | ...                  | ...                    |

### 2.3 Análise Exploratória

*(🔲 A ser preenchido pelo colega — distribuição de amostras, duração média dos áudios, taxa de amostragem, visualização de espectrogramas de exemplo por classe)*

## 3. Pipeline de Pré-processamento

O pipeline de pré-processamento converte os arquivos de áudio `.wav` em tensores prontos para alimentar as CNNs. As etapas são: **carregamento → geração do espectrograma de Mel → normalização → divisão dos dados → data augmentation**.

In [ ]:
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
import tensorflow as tf
from keras.models import Sequential, Model
from keras.layers import (Conv2D, MaxPooling2D, Dense, Dropout,
                           BatchNormalization, GlobalAveragePooling2D, Input)
from keras.applications import EfficientNetB0
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')

### 3.1 Carregamento do Áudio

Todos os arquivos de áudio são carregados com **taxa de amostragem padronizada de 22.050 Hz** — valor padrão amplamente adotado na literatura de processamento de áudio, pois captura frequências de até ~11 kHz (limite de Nyquist), cobrindo a faixa relevante para sons ambientes e de objetos do cotidiano.

Os áudios são convertidos para **mono** (canal único) e truncados ou preenchidos com zeros (*zero-padding*) para uma **duração fixa de 4 segundos**, garantindo que todos os inputs da rede tenham dimensões idênticas.

In [ ]:
SAMPLE_RATE = 22050
DURATION    = 4          # segundos
N_SAMPLES   = SAMPLE_RATE * DURATION

def load_audio(file_path: str) -> np.ndarray:
    """Carrega áudio em mono, 22050 Hz, duração fixa de 4 s."""
    audio, _ = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)
    if len(audio) < N_SAMPLES:
        audio = np.pad(audio, (0, N_SAMPLES - len(audio)))
    else:
        audio = audio[:N_SAMPLES]
    return audio

### 3.2 Geração do Espectrograma de Mel

A conversão do sinal de áudio para espectrograma de Mel é o passo central do pré-processamento. O espectrograma representa a **energia do sinal em diferentes faixas de frequência ao longo do tempo**, seguindo a escala Mel — que aproxima a percepção humana de frequências sonoras, dando mais resolução às frequências baixas onde a audição humana é mais sensível.

**Por que espectrograma de Mel e não outras representações?**

| Representação | Vantagem | Desvantagem |
|---------------|----------|-------------|
| Áudio bruto (waveform) | Sem perda de informação | CNN precisa aprender todas as features do zero; treino lento e caro |
| MFCC | Compacto, elimina redundância | Perde informação de fase e energia global; menos visual |
| **Espectrograma de Mel** | Preserva estrutura temporal e espectral; compatível com CNN 2D; eficiente | Alguma perda de informação de fase |
| CQT (Constant-Q Transform) | Boa resolução em baixas frequências | Mais custoso computacionalmente |

O espectrograma de Mel é escolhido por oferecer o melhor equilíbrio entre riqueza de informação, custo computacional e compatibilidade com CNNs.

**Parâmetros adotados:**

- `n_mels = 128`: 128 bandas de frequência na escala Mel — resolução suficiente para distinguir sons complexos sem excesso de dimensionalidade.
- `n_fft = 2048`: janela FFT de 2048 amostras (~93 ms a 22 kHz) — boa resolução em frequência.
- `hop_length = 512`: deslocamento entre janelas (~23 ms) — resolução temporal adequada para capturar transientes sonoros.
- **Conversão para dB** (`power_to_db`): comprime a escala de amplitude logaritmicamente, tornando as diferenças entre sons suaves e intensos mais tratáveis pela rede.
- **Redimensionamento para 128×128**: o espectrograma resultante (~128×173) é redimensionado para 128×128 via interpolação bilinear, padronizando o input.

In [ ]:
N_MELS     = 128
N_FFT      = 2048
HOP_LENGTH = 512
IMG_SIZE   = 128   # pixels — espectrograma quadrado

def audio_to_melspectrogram(audio: np.ndarray) -> np.ndarray:
    """Converte sinal de áudio para espectrograma de Mel em dB, shape (128, 128)."""
    mel = librosa.feature.melspectrogram(
        y=audio, sr=SAMPLE_RATE,
        n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)  # escala logarítmica
    # Redimensiona para IMG_SIZE x IMG_SIZE
    mel_resized = tf.image.resize(
        mel_db[..., np.newaxis], [IMG_SIZE, IMG_SIZE]
    ).numpy().squeeze()
    return mel_resized  # shape: (128, 128)

# Visualização de exemplo
def plot_spectrogram(spec, title='Espectrograma de Mel'):
    plt.figure(figsize=(8, 4))
    librosa.display.specshow(spec, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
                              x_axis='time', y_axis='mel')
    plt.colorbar(format='%+2.0f dB')
    plt.title(title)
    plt.tight_layout()
    plt.show()

### 3.3 Normalização

Após a geração do espectrograma, os valores são normalizados para o intervalo **[0, 1]** por meio de normalização min-max por amostra. Isso garante que a rede não seja dominada por amostras com amplitude naturalmente mais alta, além de melhorar a convergência do treinamento.

### 3.4 Divisão dos Dados

O dataset é dividido em três conjuntos de forma **estratificada por rótulo** (preservando a proporção de cada classe em treino, validação e teste):

| Conjunto   | Proporção |
|------------|----------|
| Treino     | 70%      |
| Validação  | 15%      |
| Teste      | 15%      |

In [ ]:
def normalize(spectrogram: np.ndarray) -> np.ndarray:
    s_min, s_max = spectrogram.min(), spectrogram.max()
    return (spectrogram - s_min) / (s_max - s_min + 1e-8)

def prepare_dataset(file_paths, labels):
    """
    file_paths : lista de caminhos para arquivos .wav
    labels     : array (N, 5) com rótulos multi-label one-hot
    Retorna X_train, X_val, X_test (shape: N x 128 x 128 x 1) e labels correspondentes.
    """
    spectrograms = []
    for fp in file_paths:
        audio = load_audio(fp)
        spec  = audio_to_melspectrogram(audio)
        spec  = normalize(spec)
        spectrograms.append(spec[..., np.newaxis])   # adiciona canal

    X = np.array(spectrograms)   # (N, 128, 128, 1)
    y = np.array(labels)         # (N, 5)

    X_train, X_tmp, y_train, y_tmp = train_test_split(
        X, y, test_size=0.30, random_state=42
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_tmp, y_tmp, test_size=0.50, random_state=42
    )
    return (X_train, y_train), (X_val, y_val), (X_test, y_test)

### 3.5 Data Augmentation

Com apenas ~2.290 amostras distribuídas entre 18 classes, o dataset é relativamente pequeno para o treinamento de CNNs profundas. O *data augmentation* é essencial para aumentar artificialmente a diversidade do conjunto de treino e reduzir overfitting. As técnicas adotadas operam em dois estágios:

**No sinal de áudio (antes do espectrograma):**

- **Time Shifting**: deslocar o sinal no tempo em ±20% da duração total — simula diferentes momentos de início do evento sonoro.
- **Adição de Ruído Gaussiano** (σ = 0.005): adiciona pequena perturbação ao áudio — simula variações de ambiente e microfone.

**No espectrograma (SpecAugment):**

- **Frequency Masking**: mascarar faixas horizontais aleatórias do espectrograma (até 20 bandas de frequência) — força a rede a não depender de frequências específicas.
- **Time Masking**: mascarar faixas verticais aleatórias (até 20 timesteps) — aumenta robustez a eventos de curta duração.

**Justificativa:** SpecAugment é uma técnica consolidada para modelos de áudio, com ganhos comprovados em tarefas de classificação de sons e reconhecimento de fala, mesmo em datasets pequenos.

In [ ]:
def time_shift(audio: np.ndarray, shift_max: float = 0.2) -> np.ndarray:
    shift = int(np.random.uniform(-shift_max, shift_max) * len(audio))
    return np.roll(audio, shift)

def add_noise(audio: np.ndarray, sigma: float = 0.005) -> np.ndarray:
    return audio + np.random.normal(0, sigma, len(audio))

def spec_augment(spec: np.ndarray, freq_mask=20, time_mask=20) -> np.ndarray:
    """Aplica SpecAugment: frequency masking e time masking."""
    aug = spec.copy()
    # Frequency masking
    f  = np.random.randint(0, freq_mask)
    f0 = np.random.randint(0, aug.shape[0] - f)
    aug[f0:f0+f, :] = 0
    # Time masking
    t  = np.random.randint(0, time_mask)
    t0 = np.random.randint(0, aug.shape[1] - t)
    aug[:, t0:t0+t] = 0
    return aug

def augment_audio(audio: np.ndarray) -> np.ndarray:
    """Aplica time shift e ruído com probabilidade 50% cada."""
    if np.random.rand() > 0.5:
        audio = time_shift(audio)
    if np.random.rand() > 0.5:
        audio = add_noise(audio)
    return audio

## 4. Arquitetura das Redes Neurais

Serão implementadas duas CNNs conforme especificado:
1. **CNN treinada do zero** — arquitetura própria, construída e inicializada do zero.
2. **Transfer Learning com EfficientNetB0** — backbone pré-treinado no ImageNet com cabeça de classificação customizada.

Ambas as redes recebem como input espectrogramas de Mel normalizados com shape **(128, 128, 1)** e produzem como output um vetor de 5 valores entre 0 e 1 — um por classe — representando a probabilidade de cada classe estar presente no áudio (**saída multi-label com sigmoid**).

### 4.1 Rede 1 — CNN Treinada do Zero

A arquitetura base é composta por **blocos convolucionais** (Conv2D → BatchNorm → MaxPool) seguidos de camadas densas para classificação.

**Arquitetura base (configuração de referência com 3 blocos convolucionais):**

| Camada              | Tipo                | Parâmetros                         |
|---------------------|---------------------|------------------------------------|
| Input               | —                   | (128, 128, 1)                      |
| Conv2D              | Convolucional       | 32 filtros, kernel 3×3, ReLU, same |
| BatchNormalization  | Normalização        | —                                  |
| MaxPooling2D        | Pooling             | 2×2                                |
| Conv2D              | Convolucional       | 64 filtros, kernel 3×3, ReLU, same |
| BatchNormalization  | Normalização        | —                                  |
| MaxPooling2D        | Pooling             | 2×2                                |
| Conv2D              | Convolucional       | 128 filtros, kernel 3×3, ReLU, same|
| BatchNormalization  | Normalização        | —                                  |
| MaxPooling2D        | Pooling             | 2×2                                |
| GlobalAveragePooling2D | Pooling global  | —                                  |
| Dense               | Densa               | 256 neurônios, ReLU                |
| Dropout             | Regularização       | taxa = 0.5                         |
| Dense (saída)       | Classificação       | 5 neurônios, **Sigmoid**           |

**Justificativas das escolhas arquiteturais:**

- **Blocos Conv + BN + MaxPool**: padrão consolidado para extração hierárquica de features. As camadas convolucionais iniciais capturam padrões de baixo nível (bordas e texturas no espectrograma), enquanto as camadas mais profundas detectam padrões complexos associados a cada classe sonora (harmônicos, envelopes temporais característicos).
- **BatchNormalization**: estabiliza e acelera o treinamento, permitindo taxas de aprendizado mais altas e reduzindo a sensibilidade à inicialização dos pesos.
- **GlobalAveragePooling2D**: preferido ao Flatten pois reduz drasticamente o número de parâmetros (e overfitting), calculando a média espacial de cada feature map.
- **Dropout (0.5)**: regularização crucial dado o tamanho moderado do dataset.
- **Sigmoid na saída (multi-label)**: ao contrário do Softmax — que força a distribuição de probabilidade a somar 1 (adequado para multi-classe exclusiva) — o Sigmoid aplica uma função logística independente para cada neurônio de saída. Isso permite que múltiplas classes sejam atribuídas com alta probabilidade simultaneamente, o que é obrigatório no cenário multi-label deste trabalho.

In [ ]:
def build_cnn_from_scratch(
    n_classes: int   = 5,
    n_conv_blocks: int = 3,
    filters_base: int  = 32,
    dense_units: int   = 256,
    activation: str    = 'relu'
) -> Sequential:
    """
    CNN construída do zero para classificação multi-label de espectrogramas.

    Parâmetros
    ----------
    n_conv_blocks : número de blocos convolucionais (2, 3 ou 4)
    filters_base  : número de filtros no 1º bloco; dobra a cada bloco
    dense_units   : neurônios na camada densa (128, 256 ou 512)
    activation    : função de ativação ('relu' ou 'elu')
    """
    model = Sequential(name='CNN_from_scratch')
    model.add(Input(shape=(128, 128, 1)))

    for i in range(n_conv_blocks):
        filters = filters_base * (2 ** i)
        model.add(Conv2D(filters, (3, 3), activation=activation, padding='same'))
        model.add(BatchNormalization())
        model.add(MaxPooling2D((2, 2)))

    model.add(GlobalAveragePooling2D())
    model.add(Dense(dense_units, activation=activation))
    model.add(Dropout(0.5))
    model.add(Dense(n_classes, activation='sigmoid'))

    return model

# Exemplo: instancia e exibe o sumário da configuração base
model_scratch = build_cnn_from_scratch()
model_scratch.summary()

### 4.2 Rede 2 — Transfer Learning com EfficientNetB0

A segunda rede utiliza o **EfficientNetB0** como backbone pré-treinado no ImageNet, com uma cabeça de classificação customizada.

**Por que EfficientNetB0 e não ResNet50 ou VGG16?**

| Modelo          | Parâmetros | Top-1 ImageNet | Obs. para este problema |
|-----------------|-----------|----------------|-------------------------|
| VGG16           | ~138M     | 71.3%          | Muito pesado; overfitting severo com dataset pequeno |
| ResNet50        | ~25M      | 76.0%          | Bom, mas superdimensionado para o tamanho do dataset |
| **EfficientNetB0** | ~5.3M  | **77.1%**      | Melhor acurácia com menos parâmetros — ideal para datasets menores |
| MobileNetV2     | ~3.4M     | 71.8%          | Mais leve, mas desempenho inferior |

EfficientNet escala largura, profundidade e resolução de forma balanceada (*compound scaling*), atingindo alta performance com número reduzido de parâmetros — o que é crucial para evitar overfitting dado o tamanho do dataset.

**Adaptação de canal:** O EfficientNetB0 espera inputs RGB (3 canais). O espectrograma em escala de cinza (1 canal) será replicado 3 vezes no eixo do canal antes de ser alimentado à rede.

**Estratégia de fine-tuning em duas fases:**

1. **Fase 1 — Feature Extraction** (épocas 1–10): congela todos os pesos do backbone e treina apenas a cabeça de classificação. Isso evita destruir as features ImageNet antes de adaptar a cabeça.
2. **Fase 2 — Fine-tuning** (épocas 11–50): descongela as últimas N camadas do backbone e treina com taxa de aprendizado muito reduzida (1e-5), permitindo que as camadas mais profundas se adaptem ao domínio de espectrogramas de áudio.

**Justificativa para transferência de áudio → imagem:** Embora treinado em imagens naturais, as camadas convolucionais iniciais do EfficientNetB0 detectam bordas, texturas e gradientes locais — features que são igualmente informativas em espectrogramas de Mel (por exemplo, bordas verticais correspondem a transientes sonoros; horizontais a harmônicos estacionários).

In [ ]:
def build_transfer_learning_model(
    n_classes: int       = 5,
    dense_units: int     = 256,
    activation: str      = 'relu',
    trainable_layers: int = 0
) -> Model:
    """
    Modelo com EfficientNetB0 pré-treinado + cabeça de classificação customizada.

    Parâmetros
    ----------
    dense_units      : neurônios na camada densa da cabeça (128, 256 ou 512)
    activation       : função de ativação ('relu' ou 'elu')
    trainable_layers : número de camadas finais do backbone a descongelar
                       (0 = Feature Extraction; 20 ou 50 = Fine-tuning parcial)
    """
    base_model = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=(128, 128, 3)
    )
    base_model.trainable = False  # congela backbone na Fase 1

    if trainable_layers > 0:
        for layer in base_model.layers[-trainable_layers:]:
            layer.trainable = True

    # Input: (128, 128, 1) → repete canal 3x para RGB
    inputs  = Input(shape=(128, 128, 1))
    x       = tf.keras.layers.Concatenate()([inputs, inputs, inputs])
    x       = base_model(x, training=False)
    x       = GlobalAveragePooling2D()(x)
    x       = Dense(dense_units, activation=activation)(x)
    x       = Dropout(0.5)(x)
    outputs = Dense(n_classes, activation='sigmoid')(x)

    return Model(inputs, outputs, name='EfficientNetB0_TL')

# Exemplo: instancia e exibe o sumário da configuração base
model_tl = build_transfer_learning_model()
model_tl.summary()

### 4.3 Mapeamento de Saída (Lookup Table)

A saída de ambas as redes é um vetor de 5 valores entre 0 e 1. Para obter a predição final, aplica-se um **limiar de 0.5**: valores acima do limiar indicam que aquela classe está presente no áudio. O mapeamento para nomes de classes é feito via lookup table:

In [ ]:
# Atualizar com as classes definitivas escolhidas
CLASSES = [
    'classe_1',   # ex: campainha
    'classe_2',   # ex: secador_de_cabelo
    'classe_3',   # ex: ventilador
    'classe_4',
    'classe_5'
]

def decode_prediction(output_vector: np.ndarray, threshold: float = 0.5):
    """
    Converte o vetor de saída da rede nos nomes das classes presentes.

    Exemplo:
        output_vector = [0.92, 0.08, 0.87, 0.13, 0.04]
        → ['campainha', 'ventilador']
    """
    return [cls for cls, prob in zip(CLASSES, output_vector) if prob >= threshold]

## 5. Planejamento de Experimentos

Para cada rede, serão realizados **ao menos 12 experimentos**, variando os hiperparâmetros conforme exigido pelo enunciado. Os experimentos seguem um design que isola o efeito de cada hiperparâmetro a partir de uma configuração de referência (*baseline*).

**Métricas de avaliação:**

| Métrica | Descrição |
|---------|----------|
| **F1-Score macro** | Métrica principal — média do F1 por classe, adequada para multi-label e possível desbalanceamento |
| Subset Accuracy | % de amostras onde *todas* as classes foram preditas corretamente |
| Val Loss | Binary Cross-Entropy de validação — monitora overfitting |

### 5.1 Espaço de Hiperparâmetros

| Hiperparâmetro | Variação 1 | Variação 2 | Variação 3 |
|----------------|------------|------------|------------|
| Número de blocos convolucionais / camadas fine-tuning | 2 / 0 (frozen) | **3 / 20** | 4 / 50 |
| Neurônios na camada densa | 128 | **256** | 512 |
| Função de ativação | **ReLU** | ELU | — |
| Função de perda | **Binary Cross-Entropy** | Focal Loss | — |
| Otimizador | **Adam (lr=1e-3)** | SGD+momentum (lr=1e-2, m=0.9) | — |

*Negrito = configuração baseline*

**Justificativas:**
- **ReLU vs ELU**: ReLU é padrão e eficiente; ELU suaviza gradientes para valores negativos e pode melhorar a convergência em redes profundas.
- **Binary Cross-Entropy vs Focal Loss**: Focal Loss penaliza mais fortemente os exemplos difíceis de classificar, sendo particularmente útil quando há desbalanceamento entre classes.
- **Adam vs SGD com momentum**: Adam adapta a taxa de aprendizado por parâmetro e converge mais rápido; SGD com momentum pode generalizar melhor em alguns cenários e é menos sensível a mínimos locais estreitos.

---

### 5.2 Experimentos — CNN do Zero

| Exp | Blocos Conv | Neurônios Dense | Ativação | Loss           | Otimizador | F1 (val) | Acc (val) | Observação |
|-----|-------------|-----------------|----------|----------------|------------|----------|-----------|------------|
| 1   | **2**       | 256             | ReLU     | BCE            | Adam       | —        | —         | variação profundidade |
| 2   | **3**       | 256             | ReLU     | BCE            | Adam       | —        | —         | **baseline** |
| 3   | **4**       | 256             | ReLU     | BCE            | Adam       | —        | —         | variação profundidade |
| 4   | 3           | **128**         | ReLU     | BCE            | Adam       | —        | —         | variação dense |
| 5   | 3           | **512**         | ReLU     | BCE            | Adam       | —        | —         | variação dense |
| 6   | 3           | 256             | **ELU**  | BCE            | Adam       | —        | —         | variação ativação |
| 7   | 3           | 256             | ReLU     | **Focal**      | Adam       | —        | —         | variação loss |
| 8   | 3           | 256             | ReLU     | BCE            | **SGD**    | —        | —         | variação otimizador |
| 9   | 2           | 128             | ELU      | BCE            | Adam       | —        | —         | combinação |
| 10  | 4           | 512             | ReLU     | BCE            | Adam       | —        | —         | rede maior |
| 11  | 3           | 256             | ELU      | Focal          | Adam       | —        | —         | combinação |
| 12  | 4           | 256             | ELU      | Focal          | SGD        | —        | —         | combinação |

---

### 5.3 Experimentos — Transfer Learning (EfficientNetB0)

| Exp | Fine-tuning Layers | Neurônios Dense | Ativação | Loss           | Otimizador | F1 (val) | Acc (val) | Observação |
|-----|--------------------|-----------------|----------|----------------|------------|----------|-----------|------------|
| 1   | **0 (frozen)**     | 256             | ReLU     | BCE            | Adam       | —        | —         | **baseline** |
| 2   | **20**             | 256             | ReLU     | BCE            | Adam       | —        | —         | fine-tuning parcial |
| 3   | **50**             | 256             | ReLU     | BCE            | Adam       | —        | —         | fine-tuning amplo |
| 4   | 0 (frozen)         | **128**         | ReLU     | BCE            | Adam       | —        | —         | variação dense |
| 5   | 0 (frozen)         | **512**         | ReLU     | BCE            | Adam       | —        | —         | variação dense |
| 6   | 0 (frozen)         | 256             | **ELU**  | BCE            | Adam       | —        | —         | variação ativação |
| 7   | 0 (frozen)         | 256             | ReLU     | **Focal**      | Adam       | —        | —         | variação loss |
| 8   | 0 (frozen)         | 256             | ReLU     | BCE            | **SGD**    | —        | —         | variação otimizador |
| 9   | 20                 | 256             | ELU      | BCE            | Adam       | —        | —         | combinação |
| 10  | 50                 | 512             | ReLU     | BCE            | Adam       | —        | —         | fine-tuning + rede maior |
| 11  | 20                 | 256             | ELU      | Focal          | Adam       | —        | —         | combinação |
| 12  | 50                 | 256             | ELU      | Focal          | SGD        | —        | —         | combinação |


## 6. Detalhes de Treinamento

As configurações a seguir são fixas para todos os experimentos, garantindo comparabilidade entre as redes:

| Parâmetro       | Valor             |
|-----------------|------------------|
| Épocas máximas  | 50               |
| Batch size      | 32               |
| Early Stopping  | patience = 10    |
| ReduceLROnPlateau | patience = 5, fator = 0.5 |
| Semente aleatória | 42             |

**Callbacks utilizados:**

- **EarlyStopping** (patience=10): interrompe o treinamento se a `val_loss` não melhorar por 10 épocas consecutivas, restaurando os pesos da melhor época.
- **ModelCheckpoint**: salva automaticamente o modelo com menor `val_loss` durante o treinamento, evitando perda de progresso.
- **ReduceLROnPlateau**: reduz a taxa de aprendizado por um fator de 0.5 quando a `val_loss` estagna por 5 épocas — permite convergência mais fina nas etapas finais do treinamento.

In [ ]:
def get_callbacks(model_name: str = 'model'):
    return [
        EarlyStopping(
            monitor='val_loss', patience=10,
            restore_best_weights=True, verbose=1
        ),
        ModelCheckpoint(
            f'checkpoints/{model_name}_best.keras',
            monitor='val_loss', save_best_only=True, verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=5, min_lr=1e-6, verbose=1
        )
    ]

def compile_model(model, loss='binary_crossentropy', optimizer='adam'):
    """
    loss      : 'binary_crossentropy' ou instância de Focal Loss
    optimizer : 'adam' ou tf.keras.optimizers.SGD(lr=1e-2, momentum=0.9)
    """
    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(multi_label=True, name='auc')
        ]
    )
    return model